# A planner that builds the team

Yesterday's `Finding` was a contract for *one* agent. Today we'll write a `ResearchPlan` that describes a *team* of agents — and an agent that emits one. Then we spawn the workers it described and run them as a sequential pipeline: researcher → analyst → critic.

There's no special "team" primitive in beta. The Pydantic model is the orchestration.

In [1]:
import os
from typing import Literal

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig
from autogen.beta.tools import ExaToolkit

load_dotenv()

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1",
)
exa = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))

## The schemas

Two layers. `Finding` is the researcher's contract (same as Day 2). `WorkerSpec` and `ResearchPlan` are the *planner's* contract — what the planner agent emits.

In [2]:
class Citation(BaseModel):
    title: str = Field(description="Exact title of a search result you're citing.")
    url: str = Field(description="URL of that source.")
    relevance: str = Field(description="One sentence on why this source supports your claim.")

class Finding(BaseModel):
    topic: str = Field(description="The specific aspect this finding addresses.")
    summary: str = Field(description="2-3 sentences capturing the core finding.")
    citations: list[Citation] = Field(description="Sources behind the summary.")
    novelty: float = Field(ge=0.0, le=1.0, description="0 = textbook, 1 = cutting edge.")

WorkerRole = Literal["researcher", "analyst", "critic"]

class WorkerSpec(BaseModel):
    role: WorkerRole = Field(description="What kind of work this worker does.")
    name: str = Field(description="Short lowercase identifier, no spaces.")
    prompt: str = Field(description="System prompt for this worker, focused on its role.")
    tools: list[str] = Field(description="Tool keys. Use 'exa_search' for researchers; emit [] for none.")

class ResearchPlan(BaseModel):
    topic: str = Field(description="The research topic.")
    description: str = Field(description="Why this topic is worth investigating.")
    team: list[WorkerSpec] = Field(description="Pipeline order: researcher first (to gather sources), then analyst, then critic.")

## The planner

An agent like any other — except its `response_schema` is `ResearchPlan`. Ask it for a topic, get back a typed plan.

In [3]:
planner = Agent(
    name="planner",
    prompt=(
        "You design tight 3-person research pipelines: researcher → analyst → critic, in that order. "
        "The researcher gets exa_search; analyst and critic need no tools and reason over upstream findings."
    ),
    config=config,
    response_schema=ResearchPlan,
)

plan: ResearchPlan = await (await planner.ask(
    "Plan a small team to investigate high-Tc superconductivity."
)).content()

plan_lines = [f"**{plan.topic}** — {plan.description}", ""]
for w in plan.team:
    plan_lines.append(f"**[{w.role}] {w.name}** — tools={w.tools or 'none'}")
    plan_lines.append(f"> {w.prompt}")
    plan_lines.append("")
display(Markdown("\n".join(plan_lines)))

**High-Tc superconductivity** — High-Tc superconductivity remains one of the most important open problems in condensed matter physics, with implications for theory, materials discovery, and future technology. A small research pipeline can gather current literature, synthesize competing mechanisms and materials trends, and critically assess gaps, assumptions, and unresolved questions.

**[researcher] literature_scout** — tools=['exa_search']
> You are the researcher. Use exa_search to collect up-to-date, high-quality sources on high-Tc superconductivity. Prioritize review articles, landmark experiments, recent discoveries, and authoritative overviews. Summarize key findings, major material families, proposed pairing mechanisms, and open questions, with citations and publication years.

**[analyst] synthesis_analyst** — tools=none
> You are the analyst. Using only the researcher's findings, organize the evidence into a coherent map: main material classes, dominant theoretical explanations, experimental signatures, and trends over time. Identify consensus points, points of disagreement, and the most promising directions for further study. Do not add uncited claims beyond what the researcher provided.

**[critic] skeptic_reviewer** — tools=none
> You are the critic. Review the analyst's synthesis for overreach, missing counterexamples, weak evidence, and ambiguous terminology. Stress-test the conclusions, note where the evidence is thin or contradictory, and propose specific follow-up questions or experiments that would reduce uncertainty. Do not use tools; reason only from the upstream material.


## Run the pipeline

The planner gave us role names, prompts, and tool keys. We resolve the tool keys against a local `TOOLBOX` and spawn the agents. Workers run sequentially; each one sees the previous worker's reply concatenated into its prompt. Only the researcher returns a typed `Finding` — the analyst and critic respond in markdown, since their job is to read what's already been said.

In [4]:
TOOLBOX = {"exa_search": [exa.search()]}

async def run_worker(spec: WorkerSpec, topic: str, history: str) -> Finding | str:
    typed = spec.role == "researcher"
    tools = [t for key in spec.tools for t in TOOLBOX.get(key, [])]
    worker = Agent(
        name=spec.name,
        prompt=spec.prompt,
        config=config,
        tools=tools,
        response_schema=Finding if typed else None,
    )
    user = f"Topic: {topic}\n\nContribute as {spec.role}. Respond concisely."
    if history:
        user += f"\n\nPrior workers said:\n{history}"
    reply = await worker.ask(user)
    return await reply.content() if typed else (reply.body or "")

results: list[Finding | str] = []
history = ""
for spec in plan.team:
    out = await run_worker(spec, plan.topic, history)
    results.append(out)
    history += f"\n\n[{spec.role}] {out.summary if isinstance(out, Finding) else out[:300]}"

## Render

In [5]:
lines: list[str] = []
for spec, result in zip(plan.team, results):
    lines.append(f"### {spec.name} ({spec.role})")
    if isinstance(result, Finding):
        lines.append(f"*{result.topic} — novelty={result.novelty:.2f}*")
        lines.append(result.summary)
        for c in result.citations:
            lines.append(f"- {c.title} — {c.relevance}")
    else:
        lines.append(result)
    lines.append("")
display(Markdown("\n".join(lines)))

### literature_scout (researcher)
*High-Tc superconductivity overview — novelty=0.40*
High-Tc superconductivity is now best viewed as a multi-family problem: cuprates still set the ambient-pressure record (~134–164 K under pressure), iron-based superconductors reach up to ~65 K in FeSe monolayers and ~56 K in pnictides, nickelates have emerged as a new correlated-electron platform with superconductivity in infinite-layer thin films and high-pressure Ruddlesden–Popper phases, and hydrides deliver the highest Tc but only under megabar pressure. Across these systems, the leading pairing ideas remain unconventional and material-specific: spin-fluctuation-driven d- or s±-wave pairing in cuprates/iron pnictides, more multiband and disorder-sensitive physics in nickelates, and conventional electron–phonon coupling in hydrogen-rich hydrides. Open questions center on whether one unifying mechanism exists, how pseudogap/charge order/nematicity compete or assist pairing, and how to raise Tc while improving phase purity, dimensionality control, and pressure-free stability.
- Charge Correlations in Cuprate Superconductors — 2024 review of cuprate charge order, a central competing/intertwined phenomenon tied to high-Tc pairing.
- The Physics of Pair-Density Waves: Cuprate Superconductors and Beyond — 2020 authoritative review of PDW order and its relevance to cuprates and intertwined superconducting states.
- Feshbach hypothesis of high-Tc superconductivity in cuprates — 2025 theory paper proposing a new strong-coupling pairing picture and summarizing the prevailing magnetic-pairing consensus.
- Iron pnictides and chalcogenides: a new paradigm for superconductivity — 2022 Nature review covering Fe-based families, pairing symmetry, Hund’s metals, nematicity, and quantum criticality.
- Nematicity and nematic fluctuations in iron-based superconductors — 2022 Nature Physics perspective on how nematicity and its fluctuations relate to pairing.
- Experimental Progress in Superconducting Nickelates — 2024 review summarizing the rapidly evolving nickelate experimental landscape, including thin films, charge order, and normal-state transport.
- Superconducting nickelates—a renaissance of the one-band Hubbard model — 2020 review framing nickelates in relation to cuprates and candidate pairing models.
- Superconductivity in the Parent Infinite-Layer Nickelate NdNiO2 — 2025 landmark experiment reporting superconductivity in the parent infinite-layer nickelate, sharpening open questions on intrinsic pairing vs disorder.
- Hydride superconductivity is here to stay — 2024 Nature Reviews Physics article critically reaffirms the reality of hydride superconductivity and its landmark results.
- Strategies for improving the superconductivity of hydrides under high pressure — 2024 topical review on design strategies for higher-Tc, lower-pressure hydride superconductors.
- A perspective on reducing stabilizing pressure for high-temperature superconductivity in hydrides — 2024 topical review focused on the key challenge of lowering pressure in hydride superconductors.

### synthesis_analyst (analyst)
### Coherent map of High-Tc superconductivity

#### 1) Main material classes
- **Cuprates**: still the benchmark family for ambient-pressure high Tc; record cited at **~134–164 K under pressure**.
- **Iron-based superconductors**: include **pnictides** and **FeSe-derived systems**; Tc reaches **~56 K in pnictides** and **~65 K in FeSe monolayers**.
- **Nickelates**: an emerging correlated-electron family with superconductivity in **infinite-layer thin films** and **high-pressure Ruddlesden–Popper phases**.
- **Hydrides**: highest Tc overall, but only at **megabar pressures**.

#### 2) Dominant theoretical explanations
- **Cuprates / iron pnictides**: mostly framed as **unconventional pairing** driven by **spin fluctuations**, often discussed in terms of **d-wave** or **s±-wave** symmetry.
- **Nickelates**: still unsettled; emphasized as **multiband** and **disorder-sensitive**, with material-specific correlated-electron physics.
- **Hydrides**: generally described by **conventional electron–phonon coupling**.

#### 3) Experimental signatures and control knobs
- **Pressure**: boosts Tc in cuprates and enables hydride superconductivity, but hydrides remain megabar-dependent.
- **Dimensionality**: especially important in **FeSe monolayers** and **thin-film nickelates**.
- **Phase purity / disorder sensitivity**: highlighted as key in nickelates and generally relevant across families.
- **Competing orders in cuprates**: **pseudogap, charge order, and nematicity** are central experimentally relevant features, but their role in pairing remains open.

#### 4) Trends over time
- The field has moved from seeking a single “high-Tc mechanism” toward a **multi-family framework** with **material-specific physics**.
- **Cuprates** remain the reference system, but **iron-based superconductors** and **nickelates** have expanded the landscape.
- **Hydrides** pushed Tc highest, but at the cost of extreme pressure, shifting attention to **stability and practical accessibility**.

#### 5) Consensus points
- There is **no single settled explanation** for all high-Tc systems.
- Pairing is likely **unconventional in cuprates and iron-based superconductors**.
- **Hydrides** are the clearest case for **electron–phonon-mediated superconductivity**.
- Improving Tc requires better control of **dimensionality, purity, and stability**.

#### 6) Main disagreements / open questions
- Whether a **unifying mechanism** exists across families.
- How exactly **pseudogap, charge order, and nematicity** interact with superconductivity in cuprates.
- The correct pairing description for **nickelates**, given their emerging and disorder-sensitive character.
- How to achieve **high Tc without extreme pressure**.

#### 7) Most promising directions
- **Pressure-free stabilization** of high-Tc phases.
- Better **thin-film and interface control** to exploit dimensionality effects.
- Improved **phase purity and disorder control**, especially for nickelates.
- Deeper study of **competing orders** in cuprates to clarify whether they suppress or assist pairing.

### skeptic_reviewer (critic)
The synthesis is broadly right, but it overstates coherence and underplays several caveats.

- **“Best viewed as a multi-family problem”** is reasonable, but the implied taxonomy may be too tidy. The boundaries between “cuprates,” “iron-based,” “nickelates,” and “hydrides” hide major internal diversity and overlapping physics.
- **Record Tc numbers** are a weak point: the cuprate value mixes **ambient-pressure vs under-pressure** records, which are not comparable without clarification. “~134–164 K under pressure” should not be presented as a single benchmark.
- **Nickelates** are still too unsettled to treat as an established high-Tc family on par with cuprates/iron-based systems. Superconductivity reports are limited, sample quality and phase stability remain contentious, and the pairing mechanism is not well constrained.
- **Hydrides** are correctly flagged as megabar systems, but calling them simply “conventional electron–phonon” may be too confident. Some are well described that way, but the broader family includes strong anharmonicity and structural complexity that complicate any clean label.
- The claim that **spin-fluctuation pairing** is the leading idea in cuprates and iron pnictides is plausible but not decisive; alternative or hybrid scenarios remain active, especially where nematicity, charge order, and multiorbital effects matter.
- The phrase **“one unifying mechanism”** risks overreach. Current evidence more strongly supports **family-specific mechanisms with partial commonalities** than a single universal pairing route.

**Missing counterpoints / ambiguities**
- No mention of **heavy fermions, organic superconductors, or twisted/interfacial systems**, which are relevant to “high-Tc” if the topic is mechanism rather than just record temperature.
- “High-Tc” is used ambiguously: sometimes meaning **above liquid nitrogen**, sometimes simply “unusually high among superconductors.”
- The role of **dimensionality** is asserted as a general lever, but it can also worsen disorder/sensitivity and does not universally raise Tc.

**Follow-up questions / tests**
1. Separate **ambient-pressure Tc records** from **high-pressure maxima** for each family.
2. What evidence distinguishes **pairing mechanism** from mere **correlations with competing orders**?
3. For nickelates, are superconducting signatures robust across **independent labs, substrates, and film thicknesses**?
4. In hydrides, how much of Tc is explained by **measured phonon spectra and isotope effects** versus structural stabilization?
5. Can a single theoretical framework reproduce **cuprate pseudogap**, **pnictide s± phenomenology**, and **hydride phonon-mediated pairing** without becoming vacuous?


## Up next

It runs — but every worker we spawn has its own private conversation, and we're stitching context together by hand by stuffing summaries into prompts. That's fragile: the analyst sees a 300-character snapshot, not what the researcher actually searched for, and any citation it produces is downstream of our serialisation. Tomorrow: stop manually serialising. Share one `MemoryStream` across all workers, and use an observer to capture every search hit live so we can render real hyperlinks.